# **EuroSAT V2 Pipeline: Classical ResNet18 Transfer Learning Baseline**

**NOTE:** Use T4 GPU Runtime on Google Colab
To run this notebook efficiently, you must enable a GPU accelerator in Google Colab.

* Deep learning models (like ResNet18) rely on thousands of parallel matrix multiplications
* Running this on a standard CPU will bottleneck the process and take exponentially longer time

### **Objective**
* To establish our rigorous classical performance floor.
* Initialize a deep, pre-trained ResNet18 backbone.
* Freeze the backbone and only train the final classification head $\implies$ Isolate the exact quality of linearly separable features that this deep architecture can extract.

**Execution Strategy:** We'll wrap our training execution in a loop across 3 distinct random seeds for 15 epochs each to calculate a rigorous Mean Validation Accuracy (± Standard Deviation) to ensure empirical rigor

#### **Clone git repo, setup environment and install requirements**

In [1]:
import os
import sys
import torch

# Check if the repository folder already exists on the Colab disk
if os.path.exists('/content/QML4EO-reproduction'):
    print("Repo found! Pulling latest changes from GitHub..")
    os.chdir('/content/QML4EO-reproduction')
    !git pull
else:
    print("Cloning repo for the first time..")
    os.chdir('/content')
    !git clone https://github.com/yeshapan/QML4EO-reproduction.git
    os.chdir('/content/QML4EO-reproduction')

# Append project root to sys.path so the 'src' package can be imported natively
if '/content/QML4EO-reproduction' not in sys.path:
    sys.path.append('/content/QML4EO-reproduction')

# Install required dependencies quietly
!pip install -r requirements.txt -q

# Verify device and connect to the T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nHardware utilized: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Cloning repo for the first time..
Cloning into 'QML4EO-reproduction'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 147 (delta 57), reused 122 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 13.66 MiB | 12.31 MiB/s, done.
Resolving deltas: 100% (57/57), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB

#### **Initialize Data Loaders**

In [2]:
# Import custom data loader from the src/utils folder
from src.utils.data_loader import get_eurosat_dataloaders

# Download and load the data
# (Visual verification was handled in Notebook 01, so we skip it here)
train_loader, val_loader, classes = get_eurosat_dataloaders(
    data_dir="./data",
    batch_size=64,
    img_size=224, # Changed img_size to 64 to 224
    pin_memory=True if device == "cuda" else False
)

100%|██████████| 94.3M/94.3M [00:00<00:00, 232MB/s]


Dataset loaded successfully! Total: 27000 | Train: 21600 | Val: 5400


In the above code block, we resized the image from EuroSAT's $64 \times 64$ to ResNet's $224 \times 224$
* When we stetch a $64 \times 64$ grid $224 \times 224$ grid $\rightarrow$ empty pixels are created
* PyTorch fills these empty pixels using a mathematical algorithm called Bilinear Interpolation.

#### **Establish the Classical Benchmark (ResNet18)**

We import our `ResNet18Baseline` from the `src/baselines/cnn.py` file.

The loop below will execute the model training over 3 specific seeds.
* For each seed, it resets the random state so the classification head weights are initialized differently
* This will allow us to capture the variance of the model's performance

In [3]:
import numpy as np
from src.baselines.cnn import ResNet18Baseline, train_baseline, set_seed

# Define rigorous test parameters
SEEDS = [42, 100, 2026]
EPOCHS = 10
LEARNING_RATE = 0.0001

# Containers to hold metrics for statistical analysis
all_train_losses = []
all_val_accs = []

print("\nStarting Rigorous Classical ResNet18 Training:")

for seed in SEEDS:
    print("\n")
    print(f" TRAINING SEED: {seed} ")

    # Lock the randomness for this specific run
    set_seed(seed)

    # Initialize the V2 model with the frozen ResNet18 backbone
    model = ResNet18Baseline(num_classes=len(classes))
    model = model.to(device)

    # Execute the training loop (only training the unfrozen linear head)
    history = train_baseline(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        device=device
    )

    # Save the full array of metrics for this seed
    all_train_losses.append(history['train_loss'])
    all_val_accs.append(history['val_acc'])

# Convert lists to numpy arrays to easily calculate mean and standard deviation across seeds
acc_array = np.array(all_val_accs)
loss_array = np.array(all_train_losses)

mean_acc = np.mean(acc_array, axis=0)
std_acc = np.std(acc_array, axis=0)

# Calculate and print total trainable parameters once (should be tiny compared to total model size)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n Total Trainable Parameters (Linear Head Only): {total_params:,}")

# Report the final rigorous metric
print(f" Final Classical ResNet18 Baseline: {mean_acc[-1]:.2f}% ± {std_acc[-1]:.2f}%")


Starting Rigorous Classical ResNet18 Training:


 TRAINING SEED: 42 
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]
Epoch 1/10 [Train]: 100%|██████████| 338/338 [01:00<00:00,  5.60it/s, loss=0.1652]


Epoch 1 Summary → Train Loss: 0.2227 | Val Accuracy: 97.28%


Epoch 2/10 [Train]: 100%|██████████| 338/338 [00:56<00:00,  5.97it/s, loss=0.0348]


Epoch 2 Summary → Train Loss: 0.0951 | Val Accuracy: 96.78%


...

Epoch 8 Summary → Train Loss: 0.0447 | Val Accuracy: 97.98%


Epoch 9/10 [Train]: 100%|██████████| 338/338 [00:55<00:00,  6.13it/s, loss=0.0443]


Epoch 9 Summary → Train Loss: 0.0399 | Val Accuracy: 98.50%


Epoch 10/10 [Train]: 100%|██████████| 338/338 [00:56<00:00,  6.03it/s, loss=0.0247]


Epoch 10 Summary → Train Loss: 0.0324 | Val Accuracy: 98.37%

 Total Trainable Parameters (Linear Head Only): 11,181,642
 Final Classical ResNet18 Baseline: 98.15% ± 0.31%


NOTE: Save the model weights to Google Drive to use them for HQCNN execution

In [9]:
import os
import torch

# Ensure the Google Drive folder exists
drive_model_dir = '/content/drive/MyDrive/QML4EO-reproduction/models'
os.makedirs(drive_model_dir, exist_ok=True)

# Define the exact Google Drive path
drive_save_path = os.path.join(drive_model_dir, "finetuned_resnet18_eurosat.pth")

# Save the model directly to Drive!
torch.save(model.state_dict(), drive_save_path)
print(f"\n Weights saved DIRECTLY and PERMANENTLY to Google Drive at: {drive_save_path}")


 Weights saved DIRECTLY and PERMANENTLY to Google Drive at: /content/drive/MyDrive/QML4EO-reproduction/models/finetuned_resnet18_eurosat.pth
